# Safe Gardens: Plant Leaf Disease Detection
**Automated detection and classification of plant leaf diseases using colour-space segmentation, texture feature analysis and deep learning**

ARTI 403 – Image Processing · Group 2 · Imam Abdulrahman Bin Faisal University

| No. | Student | ID | Role |
|---|---|---|---|
| 1 | Nawaf Rabee Shahbal (Leader) | 2240009025 | Project lead & pipeline architect |
| 2 | Abbas Abdulaziz Al Thunayan | 2240004112 | Pre-processing |
| 3 | Mohammed Nasser Al-Marhoon | 2240005893 | Colour segmentation |
| 4 | Faisal Abdullah Alshehri | 2240001737 | Morphology & feature analysis |
| 5 | Saad Abdulaziz Alshehri | 2240006495 | Machine learning & CNN |
| 6 | Mohammed Sameer Alajlan | 2240002025 | Evaluation & visualisation |

This notebook walks through the complete pipeline in order:

1. Dataset and stratified split
2. Pre-processing (Gaussian → bilateral → CLAHE on L\*)
3. Colour-space segmentation (HSV / CIELAB, Otsu, K-means) and morphology
4. Feature extraction (colour moments, GLCM, LBP, shape, HOG)
5. Classical classifiers (SVM, Random Forest) and ablation study
6. MobileNetV2 transfer-learning benchmark
7. Evaluation: accuracy, precision, recall, F1, ROC-AUC, IoU/Dice, confusion matrices, robustness

All reusable code lives in `src/`; the long-running steps are also available as scripts in `scripts/`.
Heavy steps (feature extraction for 11k images, CNN training) are cached: set the flags below to `True` to recompute from scratch.

In [ ]:
import os, sys, json, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt, cv2
sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
sys.path.insert(0, os.path.abspath(os.path.join('..', 'scripts')))

from config import *
from data import get_split
from preprocessing import load_rgb, preprocess, DEGRADATIONS
from segmentation import (leaf_mask, lesion_mask, leaf_mask_otsu, leaf_mask_kmeans, boundary,
                          LEAF_METHODS, iou_dice, reference_leaf_mask)
from features import extract_all, FEATURE_GROUPS

RECOMPUTE_FEATURES = False   # ~25 min on 2 CPU cores
RETRAIN_CLASSICAL  = False   # ~20 min
RETRAIN_CNN        = False   # ~90 min on CPU (8 epochs); a few minutes on a GPU
plt.rcParams['figure.dpi'] = 110

## 1. Dataset
PlantVillage (Hughes & Salathé, 2015): 38 classes, 14 crop species, 256×256 RGB. We use a seeded, class-balanced subset of up to 300 images per class (11,227 images) obtained with `scripts/download_dataset.py`, split 70/15/15 with stratification.

In [ ]:
df, classes = get_split()
print(len(df), 'images,', len(classes), 'classes')
print(df.split.value_counts().to_string())
df.groupby('class').size().rename('images').to_frame().T

In [ ]:
fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for ax, k in zip(axes.ravel(), range(0, 38, 38 // 24 + 0)):
    p = df[df.label == k].path.iloc[0]
    ax.imshow(load_rgb(p)); ax.set_title(pretty(classes[k]).replace(': ', ':\n'), fontsize=7); ax.axis('off')
plt.tight_layout(); plt.show()

## 2. Pre-processing
Gaussian smoothing (5×5, σ=1) removes sensor noise, the bilateral filter (d=9, σc=σs=50) smooths while preserving lesion edges, and CLAHE (clip 2.0, 8×8 tiles) on the L\* channel of CIELAB normalises illumination without changing colour.

In [ ]:
sample = df[df['class'] == 'Tomato___Early_blight'].sort_values('path').path.iloc[5]
img = load_rgb(sample)
out, stages = preprocess(img, return_stages=True)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for i, (name, im) in enumerate(stages.items()):
    axes[0, i].imshow(im); axes[0, i].set_title(name); axes[0, i].axis('off')
    axes[1, i].hist(cv2.cvtColor(im, cv2.COLOR_RGB2LAB)[..., 0].ravel(), 64, color='seagreen'); axes[1, i].set_title('L* histogram')
plt.tight_layout(); plt.show()

## 3. Colour-space segmentation and morphology
**Leaf mask:** K-means (k=3) on the CIELAB chromaticity plane (a\*, b\*); clusters whose centre lies in the leaf-coloured half-plane (b\*>0, chromatic) are kept, followed by closing/opening, largest connected component and hole filling.  
**Lesion mask:** K-means on (a\*, b\*) inside the leaf; the least-green cluster, gated by an Otsu threshold on a\* and a non-green hue, plus dark necrotic tissue; cleaned by opening/closing.

In [ ]:
x = preprocess(img)
lab = cv2.cvtColor(x, cv2.COLOR_RGB2LAB); hsv = cv2.cvtColor(x, cv2.COLOR_RGB2HSV)
leaf = leaf_mask(x); les = lesion_mask(x, leaf)
ov = x.copy(); ov[~leaf] = (ov[~leaf] * .3).astype(np.uint8); ov[les] = [255, 40, 40]; ov[boundary(leaf)] = [255, 255, 0]
panels = [('Pre-processed', x), ('HSV saturation', hsv[..., 1]), ('Lab a*', lab[..., 1]), ('Lab b*', lab[..., 2]),
          ('Otsu on S', leaf_mask_otsu(x)), ('K-means a*b*', leaf_mask_kmeans(x)), ('Leaf mask', leaf), ('Lesion mask', les), ('Overlay', ov)]
fig, axes = plt.subplots(1, 9, figsize=(20, 2.8))
for ax, (t, im) in zip(axes, panels):
    ax.imshow(im, cmap=None if im.ndim == 3 else 'gray'); ax.set_title(t, fontsize=9); ax.axis('off')
plt.show()
print(f'lesion area = {les.sum() / leaf.sum():.1%} of the leaf')

### 3.1 Quantitative leaf-segmentation evaluation
IoU / Dice against the 555 background-removed reference images that ship with PlantVillage (`segmented` folder).

In [ ]:
seg_csv = os.path.join(TABLE_DIR, 'segmentation_iou.csv')
if not os.path.exists(seg_csv):
    import evaluate_segmentation; evaluate_segmentation.main()
seg = pd.read_csv(seg_csv)
summary = seg.groupby(['input', 'method'])[['iou', 'dice', 'ms']].mean().round(4)
summary

In [ ]:
from IPython.display import Image, display
display(Image(os.path.join(FIG_DIR, 'fig_segmentation_pipeline.png'), width=900))
display(Image(os.path.join(FIG_DIR, 'fig_segmentation_iou.png'), width=450))
lr = json.load(open(os.path.join(TABLE_DIR, 'lesion_ratio_summary.json'))); lr

## 4. Feature extraction
Four groups per image: **colour** moments (RGB/HSV/Lab, leaf and lesion regions) + hue and a\* histograms (78), **texture** = GLCM (6 properties × 3 distances × mean/range over 4 angles) + uniform LBP (64), **shape** of the lesion map (8) and **HOG** (1,764).

In [ ]:
f = extract_all(img)
{k: v.shape for k, v in f.items()}

In [ ]:
if RECOMPUTE_FEATURES or not os.path.exists(os.path.join(CACHE_DIR, 'feat_full.npz')):
    import extract_features; sys.argv = ['x']; extract_features.main()
z = np.load(os.path.join(CACHE_DIR, 'feat_full.npz'))
{g: z[g].shape for g in FEATURE_GROUPS}

## 5. Classical classifiers
SVM (RBF; C tuned on the validation split; PCA retaining 99% variance when HOG is included) and Random Forest (500 trees). E1 compares feature groups, E2 ablates pre-processing and the leaf mask, E3 tests robustness to degraded images.

In [ ]:
res_path = os.path.join(TABLE_DIR, 'classical_results.json')
if RETRAIN_CLASSICAL or not os.path.exists(res_path):
    import train_classical; train_classical.main()
res = json.load(open(res_path))
fg = pd.DataFrame(res['feature_groups'])[['features', 'dims', 'classifier', 'accuracy', 'macro_f1', 'weighted_f1']]
fg.round(4)

In [ ]:
print('Best feature set:', res['best_feature_set'])
pd.DataFrame(res['ablation'])[['setting', 'accuracy', 'macro_f1']].round(4)

## 6. MobileNetV2 transfer learning
ImageNet-pretrained MobileNetV2 fine-tuned end-to-end (224×224, AdamW, cosine schedule, label smoothing 0.1, augmentation). The checkpoint with the best validation macro-F1 is kept.

In [ ]:
from cnn import train as train_cnn, load_trained
if RETRAIN_CNN:
    import train_cnn as tc; sys.argv = ['x', '--epochs', '8']; tc.main()
hist = pd.DataFrame(json.load(open(os.path.join(TABLE_DIR, 'cnn_history.json'))))
hist.round(4)

In [ ]:
display(Image(os.path.join(FIG_DIR, 'fig_cnn_training.png'), width=900))

## 7. Results and evaluation

In [ ]:
import make_figures
make_figures.main()
pd.read_csv(os.path.join(TABLE_DIR, 'main_results.csv')).round(4)

In [ ]:
for f in ['fig_feature_groups.png', 'fig_per_class_f1.png', 'fig_confusion_svm.png', 'fig_confusion_cnn.png',
          'fig_degradations.png', 'fig_robustness.png', 'fig_lesion_ratio.png']:
    display(Image(os.path.join(FIG_DIR, f), width=800))

In [ ]:
pd.read_csv(os.path.join(TABLE_DIR, 'robustness_macro_f1.csv')).round(4)

In [ ]:
eff = json.load(open(os.path.join(TABLE_DIR, 'cnn_efficiency.json'))); eff

## 8. Inference on a single leaf
End-to-end prediction with both pipelines for one test image.

In [ ]:
import joblib, torch
from cnn import EVAL_TF
from PIL import Image as PILImage
svm = joblib.load(os.path.join(MODEL_DIR, 'svm_best.joblib'))
best_groups = {'Colour': ['colour'], 'Texture (GLCM+LBP)': ['texture'], 'Shape': ['shape'], 'HOG': ['hog'],
               'Colour+Texture': ['colour', 'texture'], 'Colour+Texture+Shape': ['colour', 'texture', 'shape'],
               'All (C+T+S+HOG)': ['colour', 'texture', 'shape', 'hog']}[res['best_feature_set']]
cnn_model = load_trained(len(classes), os.path.join(MODEL_DIR, 'mobilenetv2_best.pt'))
row = df[df.split == 'test'].sample(1, random_state=7).iloc[0]
rgb = load_rgb(row.path)
feat = extract_all(rgb); xvec = np.hstack([feat[g] for g in best_groups])[None]
svm_pred = classes[int(svm.decision_function(xvec).argmax())]
with torch.no_grad():
    cnn_pred = classes[int(cnn_model(EVAL_TF(PILImage.fromarray(rgb))[None]).argmax())]
plt.imshow(rgb); plt.axis('off'); plt.title(f'True: {pretty(row["class"])}\nSVM: {pretty(svm_pred)} | CNN: {pretty(cnn_pred)}'); plt.show()

## 9. Conclusion
Colour-space segmentation (K-means on Lab a\*b\* + morphology) isolates the leaf with high overlap against the PlantVillage reference masks, and the hand-crafted colour/texture/shape/HOG descriptors give a strong, interpretable classical baseline. Fine-tuned MobileNetV2 is the most accurate model, while the pre-processing stage measurably improves the classical pipeline under noise and poor illumination. See the paper in `paper/` for the full discussion.